In [ ]:
import nibabel as nib
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import matplotlib.pyplot as plt
import numpy as np
import pydicom
import pandas as pd
from ipywidgets import Dropdown, IntSlider, VBox, Output, Label, interactive
from IPython.display import display, clear_output
from datetime import datetime

from utils.dcm2niix import *
from utils.selection import select_modalities

# Project data path
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'
APP_DATA_PATH = '/home/bule/berner_daten_analyse/apps/app_manual_preselection'
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_third_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors_selection.log')


# Preselectinon by hakim martin and julian

use the niftis which are succesfully transforemed from dicoms and save it in a .csv

In [ ]:
dwi_t1_df_3=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_merged.csv"))
dwi_t1_df_3.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_3.head()
len(dwi_t1_df_3)

dwi_t1_df_3.groupby('Patient_ID').nunique().head()

In [ ]:
#dwi
df_dwi = dwi_t1_df_3[dwi_t1_df_3['Modality']=='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_dwi['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_dwi= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_dwi[0])

# t1
df_t1= dwi_t1_df_3[dwi_t1_df_3['Modality']!='DWI']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_t1['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])
target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_t1= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_t1[0])

all_niftis=list_nifti_files(NII_TARGET_PATH)
df_modality = pd.DataFrame({
    'Path_Nifti': target_nii_sequences_dwi+target_nii_sequences_t1,
    'Modality': ['dwi'] * len(target_nii_sequences_dwi)+['anatomic'] * len(target_nii_sequences_t1),
})
df_modality['join_crit']=[ os.path.join('/',*path.split('/')[6:]) for path in df_modality['Path_Nifti']] # just  to get the modality
df_modality.head()

In [ ]:
# check for duplicated  select all created niftis -> some didnt work
data=[]
for path in all_niftis:
    patient_id = int(path.split("/")[6])
    study_id = path.split("/")[7]
    sequence_id = path.split("/")[8]
    full_path = path
    data.append({'Patient_ID':patient_id, 'Study_ID':study_id,'Sequence_ID':sequence_id ,'Full_Path_Nifti':full_path})
    
nii_df=pd.DataFrame(data)
nii_df['join_crit']=[ os.path.join('/',*path.split('/')[6:-1]) for path in nii_df['Full_Path_Nifti']]
nii_df=nii_df.merge(df_modality, on='join_crit', how='left')

# len(nii_df[['Sequence_ID','Study_ID']].unique()) 
duplicates = nii_df[nii_df.duplicated(subset=['Study_ID', 'Sequence_ID'], keep=False)]
print(duplicates.head(1))

nii_df.drop_duplicates(subset=['Sequence_ID'], keep='first', inplace=True)
nii_df.info()

In [ ]:
nii_df = nii_df.sort_values(by='Patient_ID')
nii_df['Date'] = nii_df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))
nii_df

In [ ]:
## save to df for skullstripping 2 sequences per patient
nii_df = nii_df.sort_values(by='Patient_ID')
nii_df['Date'] = nii_df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))

only_two_df = nii_df.groupby('Patient_ID', group_keys=False).apply(select_modalities)
only_two_df['Shape']=only_two_df['Full_Path_Nifti'].apply(lambda x: nib.load(x).shape)

only_two_df.reset_index(drop=True, inplace=True)
only_two_df['Modality'].value_counts()

In [11]:

only_two_df.to_csv(os.path.join(ANALYSIS_PATH, 'only2_third_try_df.csv'), index=False)